In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
import rapids_singlecell as rsc
import cupy as cp
import warnings
import anndata as ad
import pandas as pd
import numpy as np
import os
import cupyx.scipy.sparse as cusparse
import scipy.sparse as sp

warnings.filterwarnings("ignore")

In [ ]:
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator

rmm.reinitialize(
    managed_memory=True,  # Allows oversubscription
    pool_allocator=False,  # default is False
    devices=[0],  # GPU device IDs to register. By default registers only GPU 0.
)
cp.cuda.set_allocator(rmm_cupy_allocator)

In [ ]:
proseg_outputs = "path/to/your/proseg/output/directory"
global_clustering_outputs = "path/to/your/output/directory"

#Please make a list of Xenium bundle directory name like:

section_ids = ['output-XETG00130__0076861__Normal_lung__20251122__033042', 
               'output-XETG00130__0076861__tkbadeno1__20251122__033041', 
               'output-XETG00130__0076861__tkbadeno2__20251122__033042', 
               'output-XETG00130__0076861__tkbadeno3__20251122__033042', 
               'output-XETG00130__0076861__tkbadeno4__20251122__033042', 
               'output-XETG00130__0076861__tkbadeno5__20251122__033042', 
               'output-XETG00130__0076861__tkbadeno6__20251122__033042', 
               'output-XETG00130__0076861__tkbadeno7__20251122__033042', 
               'output-XETG00130__0076862__Normal_lung__20251122__033042', 
               'output-XETG00130__0076862__tkbadeno8__20251122__033042', 
               'output-XETG00130__0076862__tkbadeno9__20251122__033042', 
               'output-XETG00130__0076862__tkbadeno10__20251122__033042', 
               'output-XETG00130__0076862__tkbadeno11__20251122__033042', 
               'output-XETG00130__0076862__tkbadeno12__20251122__033042', 
               'output-XETG00130__0076862__tkbadeno13__20251122__033042', 
               'output-XETG00130__0076862__tkbadeno14__20251122__033042', 
               'output-XETG00130__0076921__Normal_lung__20251203__041512', 
               'output-XETG00130__0076921__tkbadeno15__20251203__041512', 
               'output-XETG00130__0076921__tkbadeno16__20251203__041512', 
               'output-XETG00130__0076921__tkbadeno17__20251203__041512', 
               'output-XETG00130__0076922__Normal_lung__20251203__041512', 
               'output-XETG00130__0076922__tkbadeno18__20251203__041512', 
               'output-XETG00130__0076922__tkbadeno19__20251203__041512', 
               'output-XETG00130__0076922__tkbadeno20__20251203__041512']



In [ ]:
for ss in range(len(section_ids)):
    print(section_ids[ss][18:-18])

In [ ]:
Batch_list = []

for section_id in section_ids:
    print(section_id[18:-18])
    explorer_path = proseg_outputs + section_id[18:-18] + ".explorer"
    h5ad_path = explorer_path + "/adata.h5ad"
    adata = sc.read_h5ad(h5ad_path)
    sc.pp.filter_cells(adata, min_counts=100)
    sc.pp.filter_cells(adata, min_genes=30)
    sc.pp.normalize_total(adata, target_sum=500)
    sc.pp.log1p(adata)
    Batch_list.append(adata)

In [ ]:
section_names = [s[18:-18] for s in section_ids]
section_names

In [ ]:
integrated = ad.concat(
    Batch_list,
    join="inner",
    label="batch",
    keys=section_names,
    index_unique="-"
)

In [ ]:
if sp.issparse(integrated.X):
    integrated.X = integrated.X.astype(np.float32)    
else:
    integrated.X = integrated.X.astype(np.float32, copy=False)

rsc.get.anndata_to_GPU(integrated)

In [ ]:
rsc.get.anndata_to_CPU(integrated)
sc.pp.pca(integrated, 
           random_state=7, 
           use_highly_variable=False)
rsc.get.anndata_to_GPU(integrated)

In [ ]:
rsc.pp.harmony_integrate(integrated, key = 'batch')

In [ ]:
rsc.pp.neighbors(integrated, n_neighbors=15, use_rep="X_pca_harmony", random_state=7)

In [ ]:
rsc.tl.leiden(integrated, resolution=1.0, key_added="leiden", random_state=7)

In [ ]:
integrated.obs['leiden']

In [ ]:
rsc.tl.umap(integrated, min_dist=0.5, spread=1.0, random_state=7)
sc.pl.umap(integrated, color=['batch'])
sc.pl.umap(integrated, color=["leiden"])

In [ ]:
clusters = integrated.obs['leiden'].astype(str)
colors = integrated.uns['leiden_colors']

color_map = dict(zip(sorted(clusters.unique()), colors))
color_map

In [ ]:
seurat_colors = [
    "#F8766D", "#D89000", "#A3A500", "#39B600", "#00BF7D",
    "#00BFC4", "#00B0F6", "#619CFF", "#9590FF", "#C77CFF",
    "#E76BF3", "#FF62BC", "#FF61C3", "#FF9DA7", "#F564E3",
    "#00BA38", "#00C08B", "#00A9FF", "#0072B2", "#56B4E9",
    "#009E73", "#E69F00", "#CC79A7", "#999999", "#7CAE00",
    "#00BFC4", "#C77CFF", "#F8766D", "#00BA38", "#619CFF"
]

In [ ]:
seurat_color_map = {
    str(i): seurat_colors[i % len(seurat_colors)]
    for i in range(len(color_map))
}

In [ ]:
integrated.obs["color"] = clusters.map(seurat_color_map)

In [ ]:
integrated.obs["leiden"] = integrated.obs["leiden"].astype("category")
cats = integrated.obs["leiden"].cat.categories
integrated.uns["leiden_colors"] = seurat_colors[:len(cats)]

In [ ]:
import pickle

pickle_path = global_clustering_outputs + "/proseg_harmony_Batch_list.pkl"
with open(pickle_path, "wb") as f:
    pickle.dump(Batch_list, f)

In [ ]:
sc.pl.umap(integrated, color=['batch'])
sc.pl.umap(integrated, color=["leiden"])

In [ ]:
plot_name = "Harmony_UMAP_Sample"

umap = sc.pl.umap(integrated, color=['batch'], title="", show=False)

fig = plt.gcf()
fig.subplots_adjust(top=0.88)
fig.suptitle(plot_name, y=0.95, fontsize=20)

fig.savefig(global_clustering_outputs + "/" + plot_name + ".tiff", bbox_inches="tight", dpi=800)
plt.close(fig)

In [ ]:
plot_name = "Harmony_UMAP_Leiden_clustering"

umap = sc.pl.umap(integrated, color=['leiden'], title="", show=False)

fig = plt.gcf()
fig.subplots_adjust(top=0.88)
fig.suptitle(plot_name, y=0.95, fontsize=20)

fig.savefig(global_clustering_outputs + "/" + plot_name + ".tiff", bbox_inches="tight", dpi=800)
plt.close(fig)

In [ ]:
h5ad_path = global_clustering_outputs + "/proseg_harmony_integrated.h5ad"
integrated.write_h5ad(h5ad_path)

In [ ]:
marker_genes = {
    "Epithelium": ["EPCAM", "CDH1", "KRT5", "KRT8"],
    "Endothelium": ["PECAM1", "ERG", "CD31", "CD34", "CLDN5"],
    "Fibroblast": ["COL1A1", "DCN", "LUM"],
    "CAF": ["COL11A1", "CXCL12", "FAP", "POSTN"], 
    "Smooth muscle": ["ACTA2", "MYH11", "TAGLN"],
    "Pericyte": ["LAMC3", "RGS5", "PDGFRB"],
    "Inflammatory cells": ["PTPRC"],
    "T/NK-cell": ["CD3E", "GZMA"],
    "B-cell": ["BANK1", "CXCR5"],
    "Plasma cell": ["MZB1", "TNFRSF17"],
    "Macrophage": ["MARCO", "MS4A4A", "CD68"],
    "Dendritic cell": ["GZMB", "CLEC10A"],
    "Mast cell": ["RGS13", "KIT"],
    "Granulocytes": ["TREM1", "APOE"],
    "Proliferation": ["MKI67", "TOP2A", "MCM2", "PCNA"]
}

In [ ]:
marker_genes_in_data = dict()
for ct, markers in marker_genes.items():
    markers_found = list()
    for marker in markers:
        if marker in integrated.var.index:
            markers_found.append(marker)
    marker_genes_in_data[ct] = markers_found

In [ ]:
marker_genes_in_data

In [ ]:
population = [
    "Epithelium",
    "Endothelium",
    "Fibroblast",
    "CAF", 
    "Smooth muscle",
    "Pericyte",
    "Inflammatory cells",
    "T/NK-cell",
    "B-cell",
    "Plasma cell",
    "Macrophage",
    "Dendritic cell",
    "Mast cell",
    "Granulocytes",
    "Proliferation"
]

In [ ]:
canonical_markers = {
    ct: [m for m in ct_markers if m in integrated.var.index]
    for ct, ct_markers in marker_genes.items()
    if ct in population
}

In [ ]:
rsc.get.anndata_to_CPU(integrated)

plot_name = "leiden_dotplot"

dp = sc.pl.dotplot(
    integrated, 
    groupby="leiden", 
    title=None, 
    var_names=canonical_markers, 
    standard_scale="var", 
    figsize=(15, 20), 
    var_group_rotation=45, 
    show=False)

fig = plt.gcf()
fig.subplots_adjust(top=0.88)
fig.suptitle(plot_name, y=0.95, fontsize=20)
fig.savefig(global_clustering_outputs + "/" + plot_name + ".tiff", bbox_inches="tight", dpi=500)
plt.close(fig)

cluster_counts = integrated.obs["leiden"].value_counts()
cluster_ratios = cluster_counts / cluster_counts.sum() * 100
print(cluster_ratios)

In [ ]:
cluster_ratios.to_csv(global_clustering_outputs + "/cluster_ratios.csv", index=True)

In [ ]:
for ss in range(len(section_ids)):
    adata = integrated[integrated.obs['batch'] == section_ids[ss][18:-18]].copy()
    df = adata.obs[["cell_id", "leiden", "color"]].copy()
    df.columns = ["cell_id", "group", "color"]
    path = os.path.join(global_clustering_outputs + "/" +  
                        section_ids[ss][18:-18] + 
                        "_leiden.csv")
    df.to_csv(path, index=False)